In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "DimRed"
feature_cols = ["x1", "x2", "x3", "x4", "x5"]
df = pd.read_excel(file_path, sheet_name=sheet_name)
X = df[feature_cols].to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {
    "pca_n_components": 2,    # PCA降到几维
    "tsne_n_components": 2,   # TSNE降到几维
    "tsne_perplexity": 30.0,  # TSNE困惑度
    "umap_n_components": 2,   # UMAP维度
    "umap_n_neighbors": 15,   # UMAP邻居数
    "umap_min_dist": 0.1      # UMAP最小距离
}

X_pca = PCA(n_components=params["pca_n_components"]).fit_transform(X)
X_tsne = TSNE(
    n_components=params["tsne_n_components"],
    perplexity=params["tsne_perplexity"],
    random_state=42
).fit_transform(X)

import umap
X_umap = umap.UMAP(
    n_components=params["umap_n_components"],
    n_neighbors=params["umap_n_neighbors"],
    min_dist=params["umap_min_dist"],
    random_state=42
).fit_transform(X)

print(X_pca.shape, X_tsne.shape, X_umap.shape)


In [ ]:
"""
PCA、T-SNE、UMAP 降维

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "PCA、T-SNE、UMAP 降维.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
PCA_COMPONENTS = 2  # TODO: 请填写[PCA 主成分数]，说明：通常取 2、3 或累计贡献率达标的数量。
RUN_TSNE = True  # TODO: 请填写[是否运行 t-SNE]，说明：样本很多时会较慢。
RUN_UMAP = False  # TODO: 请填写[是否运行 UMAP]，说明：需要安装 umap-learn。
TSNE_PERPLEXITY = 30  # TODO: 请填写[t-SNE 困惑度]，说明：一般 5 到 50，且小于样本数。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    X = StandardScaler().fit_transform(data.select_dtypes(include=[np.number]))
    pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
    pca_result = pca.fit_transform(X)
    result = pd.DataFrame(pca_result, columns=[f"PC{i+1}" for i in range(PCA_COMPONENTS)])
    print("PCA 方差解释率:", pca.explained_variance_ratio_)

    if RUN_TSNE:
        tsne = TSNE(n_components=2, perplexity=TSNE_PERPLEXITY, random_state=RANDOM_STATE, init="pca")
        result[["TSNE1", "TSNE2"]] = tsne.fit_transform(X)
    # if RUN_UMAP:
    #     import umap
    #     reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE)
    #     result[["UMAP1", "UMAP2"]] = reducer.fit_transform(X)
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result.head())


if __name__ == "__main__":
    df = load_data()
    run_model(df)
